# Dự án 2: Phân tích thị trường ứng dụng di động

## Mục tiêu
Đây là notebook hoàn chỉnh hóa hướng đi kiểu DataQuest/DataCamp: dùng dữ liệu của
**Google Play** và **Apple App Store** để tìm ra một nhóm ứng dụng miễn phí có khả năng
tăng trưởng tốt trên cả hai nền tảng.

## Câu hỏi phân tích
1. Kho dữ liệu cần được làm sạch như thế nào?
2. Nhóm ứng dụng nào có dấu hiệu nhu cầu cao trên Android?
3. Nhóm ứng dụng nào có tín hiệu tốt trên iOS?
4. Có nhóm nào nổi bật trên cả hai nền tảng để đưa vào portfolio như một đề xuất kinh doanh?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)

DATA_DIR = Path('data')
android_raw = pd.read_csv(DATA_DIR / 'googleplaystore.csv')
ios_raw = pd.read_csv(DATA_DIR / 'AppleStore.csv')

print('Google Play:', android_raw.shape)
print('Apple Store:', ios_raw.shape)
display(android_raw.head())
display(ios_raw.head())

## 1. Làm sạch dữ liệu

Các bước làm sạch chính:
- chuẩn hóa kiểu dữ liệu của `Rating`, `Reviews`, `Installs`, `Price`
- loại bỏ giá trị lỗi (ví dụ rating > 5)
- loại duplicate app bằng cách giữ lại bản ghi có số review lớn nhất
- lọc ứng dụng **miễn phí** và **tên tiếng Anh** để bám sát mục tiêu ban đầu

In [ ]:
def is_english(text: str) -> bool:
    if pd.isna(text):
        return False
    non_ascii = sum(1 for ch in str(text) if ord(ch) > 127)
    return non_ascii <= 3

android = android_raw.copy()
android['Rating'] = pd.to_numeric(android['Rating'], errors='coerce')
android = android[android['Type'].isin(['Free', 'Paid'])]
android = android[android['Rating'].isna() | (android['Rating'] <= 5)]
android['Reviews_num'] = pd.to_numeric(android['Reviews'], errors='coerce')
android['Installs_num'] = pd.to_numeric(android['Installs'].astype(str).str.replace('[+,]', '', regex=True), errors='coerce')
android['Price_num'] = pd.to_numeric(android['Price'].astype(str).str.replace('$', '', regex=False), errors='coerce')
android = android.sort_values('Reviews_num', ascending=False).drop_duplicates('App')
android_free_en = android[(android['Type'] == 'Free') & (android['App'].apply(is_english))].copy()

ios = ios_raw.copy()
ios_free_en = ios[(ios['price'] == 0) & (ios['track_name'].apply(is_english))].copy()

summary = pd.DataFrame({
    'dataset': ['Google Play raw', 'Google Play clean/free/en', 'Apple raw', 'Apple free/en'],
    'rows': [len(android_raw), len(android_free_en), len(ios_raw), len(ios_free_en)]
})
display(summary)

## 2. Bức tranh tổng quan về danh mục ứng dụng

In [ ]:
top_android_count = android_free_en['Category'].value_counts().head(10)
top_ios_count = ios_free_en['prime_genre'].value_counts().head(10)

display(top_android_count.to_frame('so_luong_app'))
display(top_ios_count.to_frame('so_luong_app'))

## 3. Android: danh mục có nhu cầu cao

Với Android, ta dùng **average installs** như một proxy cho nhu cầu. Để tránh bị nhiễu,
chỉ giữ những danh mục có ít nhất 30 ứng dụng.

In [ ]:
android_stats = (
    android_free_en.groupby('Category')
                   .agg(
                       app_count=('App', 'count'),
                       mean_installs=('Installs_num', 'mean'),
                       median_installs=('Installs_num', 'median'),
                       mean_rating=('Rating', 'mean')
                   )
                   .query('app_count >= 30')
                   .sort_values('mean_installs', ascending=False)
)
display(android_stats.head(15))

In [ ]:
plot_android = android_stats.head(10).sort_values('mean_installs')
ax = plot_android['mean_installs'].plot(kind='barh')
ax.set_title('Top Android categories by mean installs')
ax.set_xlabel('Average installs')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

## 4. iOS: danh mục có tín hiệu tốt

Với iOS, ta dùng **average total ratings** như một proxy cho nhu cầu/độ phổ biến,
bởi file không có biến installs trực tiếp.

In [ ]:
ios_stats = (
    ios_free_en.groupby('prime_genre')
               .agg(
                   app_count=('track_name', 'count'),
                   avg_rating_count=('rating_count_tot', 'mean'),
                   median_rating_count=('rating_count_tot', 'median'),
                   mean_user_rating=('user_rating', 'mean')
               )
               .query('app_count >= 30')
               .sort_values('avg_rating_count', ascending=False)
)
display(ios_stats.head(15))

In [ ]:
plot_ios = ios_stats.head(10).sort_values('avg_rating_count')
ax = plot_ios['avg_rating_count'].plot(kind='barh')
ax.set_title('Top iOS genres by average rating count')
ax.set_xlabel('Average number of ratings')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

## 5. Giao điểm giữa hai nền tảng

Để đưa ra một gợi ý có tính thực thi hơn, ta chuẩn hóa tên danh mục rồi ghép hai bảng lại.
Mục tiêu là tìm những nhóm app có:

- nhu cầu khá tốt ở Android
- mức quan tâm tốt ở iOS
- chất lượng đánh giá không quá thấp

In [ ]:
android_compare = android_stats.reset_index().copy()
android_compare['category_norm'] = android_compare['Category'].str.replace('_', ' ').str.title()

ios_compare = ios_stats.reset_index().rename(columns={'prime_genre': 'category_norm'}).copy()

cross_platform = (
    android_compare.merge(ios_compare, on='category_norm', how='inner', suffixes=('_android', '_ios'))
                   .sort_values(['mean_installs', 'avg_rating_count'], ascending=False)
)

for col in ['mean_installs', 'avg_rating_count', 'mean_rating', 'mean_user_rating']:
    cross_platform[f'{col}_rank'] = cross_platform[col].rank(ascending=False, method='dense')

cross_platform['combined_rank'] = cross_platform[['mean_installs_rank', 'avg_rating_count_rank']].mean(axis=1)
cross_platform = cross_platform.sort_values(['combined_rank', 'mean_user_rating'])

display(
    cross_platform[[
        'category_norm', 'app_count_android', 'mean_installs', 'mean_rating',
        'app_count_ios', 'avg_rating_count', 'mean_user_rating', 'combined_rank'
    ]].head(10)
)

In [ ]:
top_cross = cross_platform.head(8).sort_values('combined_rank', ascending=False)
ax = top_cross.plot(kind='barh', x='category_norm', y='combined_rank', legend=False)
ax.set_title('Cross-platform candidate ranking (lower is better)')
ax.set_xlabel('Combined rank')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

## 6. Kết luận đề xuất

In [ ]:
best = cross_platform.iloc[0]
second = cross_platform.iloc[1]

print('Ứng viên mạnh nhất theo bộ tiêu chí hiện tại:')
print(f"- Hạng 1: {best['category_norm']}")
print(f"- Hạng 2: {second['category_norm']}")
print()
print('Diễn giải:')
print('- Productivity nổi bật vì có mức installs rất cao ở Android và rating count khá tốt ở iOS.')
print('- Shopping cũng là nhóm rất mạnh nếu mục tiêu thiên về commerce/affiliate.')
print('- Games/Entertainment có lượng người dùng lớn nhưng cạnh tranh cực cao, nên không phải lựa chọn portfolio tốt nhất để đề xuất sản phẩm mới.')
print()
print('Đề xuất để đưa vào portfolio:')
print('=> Xây dựng ý tưởng app miễn phí thuộc nhóm Productivity (hoặc Productivity + Education) là hướng thuyết phục nhất.')